In [1]:
import serial
import serial.tools.list_ports
import time
import cmd
from IPython.display import clear_output

In [2]:
# Initialize
ports = serial.tools.list_ports.comports()
for p in ports:
	# Arduino Nano Every 通常 VID=0x2341, PID=0x005E（或0x024E）
	if p.vid == 0x2341 and p.pid in [0x005E, 0x024E]:
		port =  p.device
	# 部分系统可能没有VID/PID，可匹配描述
	if "Arduino" in p.description or "Nano Every" in p.description:
		port = p.device
RMC = serial.Serial(port = port, baudrate = 115200, timeout = 0.2)


In [ ]:
class RelayBox:
    def __init__(self, port=None, baudrate=115200, timeout=2):
        """
        初始化串口连接
        :param port: 串口号，如 'COM3' 或 '/dev/ttyACM0'；若为None则自动检测
        :param baudrate: 波特率，需与Arduino一致
        :param timeout: 串口读取超时（秒）
        """
        if port is None:
            port = self._find_arduino_port()
            if port is None:
                raise RuntimeError("未找到Arduino串口，请手动指定端口")
        self.ser = serial.Serial(port, baudrate, timeout=timeout)
        time.sleep(2)  # 等待Arduino复位
        self.ser.reset_input_buffer()

    @staticmethod
    def _find_arduino_port():
        """自动查找Arduino Nano Every的串口（根据VID/PID或描述）"""
        ports = serial.tools.list_ports.comports()
        for p in ports:
            # Arduino Nano Every 通常 VID=0x2341, PID=0x005E（或0x024E）
            if p.vid == 0x2341 and p.pid in [0x005E, 0x024E]:
                return p.device
            # 部分系统可能没有VID/PID，可匹配描述
            if "Arduino" in p.description or "Nano Every" in p.description:
                return p.device
        return None

    def send_command(self, cmd):
        """发送命令并返回响应（忽略OK/ERROR前缀，只返回数据）"""
        self.ser.write((cmd + '\n').encode())
        line = self.ser.readline().decode().strip()
        return line

    def relay_on(self, channel):
        """开启指定通道（2~6）"""
        if channel < 2 or channel > 6:
            raise ValueError("通道号必须为2~6")
        resp = self.send_command(f"ON{channel}")
        if resp.startswith("OK"):
            return True
        else:
            raise RuntimeError(f"开启失败: {resp}")

    def relay_off(self, channel):
        """关闭指定通道"""
        if channel < 2 or channel > 6:
            raise ValueError("通道号必须为2~6")
        resp = self.send_command(f"OFF{channel}")
        if resp.startswith("OK"):
            return True
        else:
            raise RuntimeError(f"关闭失败: {resp}")

    def read_temperature(self):
        """读取温度值（摄氏度）"""
        resp = self.send_command("TEMP")
        try:
            return float(resp)
        except ValueError:
            raise RuntimeError(f"温度解析失败: {resp}")
    def read_raw_temperature(self):
        """读取温度值（摄氏度）"""
        resp = self.send_command("RAW_TEMP")
        try:
            return float(resp)
        except ValueError:
            raise RuntimeError(f"温度解析失败: {resp}")

    def get_status(self):
        """获取所有通道状态，返回字典 {channel: 'ON'/'OFF'}"""
        resp = self.send_command("STATUS")
        # 解析 "2:ON 3:OFF 4:ON ..."
        status = {}
        parts = resp.split()
        for item in parts:
            if ':' in item:
                ch, state = item.split(':')
                status[int(ch)] = state
        return status

    def close(self):
        self.ser.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

box = RelayBox()
print("成功连接到Arduino")

In [ ]:
# Continously monitoring


try:
	while True:
		clear_output(wait=True)
		RMC.write(('TEMP\n').encode())
		line = RMC.readline().decode().strip()
		print(f"Calibrated Temperature: {line}")
		#print(f"Raw Temperature: {RMC.read_raw_temperature()}")
		time.sleep(0.5)
except KeyboardInterrupt:
	print("Stopped by user")

Calibrated Temperature: 25.1


In [ ]:
self.ser.write((cmd + '\n').encode())
		line = self.ser.readline().decode().strip()

In [3]:
cmd = f"OFF{6}"
RMC.write((cmd + '\n').encode())

5

In [ ]:
cmd = f"TEMP"
RMC.write((cmd + '\n').encode())

In [ ]:
RMC = serial.Serial(port = port, baudrate = 115200, timeout = 0.2)

In [ ]:
RMC.close()

In [ ]:
box = RelayBox()

In [ ]:
box.close()

In [ ]:
box = RelayBox()

from IPython.display import clear_output
while True:
    
    print(f"{box.read_temperature()} | {box.read_raw_temperature()}")
    time.sleep(0.1)
    clear_output(wait=True)

box.close()

In [ ]:
box.close()